# CONFIGURACIÓN INICIAL & SPARK SESSION

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, LongType, DoubleType, BooleanType, TimestampType, StringType
)
from pyspark.sql.functions import col, when, from_unixtime, date_format, sum as spark_sum
from pyspark.sql.window import Window
import os

spark = (
    SparkSession.builder
    .appName("ProyectoSello-Crypto-Regresion")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 15:17:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
PATH_DATA = "/opt/data"
PATH_ARTIFACTS = "/opt/artifacts"
os.makedirs(PATH_DATA, exist_ok=True)
os.makedirs(PATH_ARTIFACTS, exist_ok=True)

# Integración de fuentes, calidad de datos y arquitectura Medallion (Bronze/Silver/Gold)

# 1. CAPA BRONZE: Lectura de fuentes crudas con ESQUEMA EXPLÍCITO

## Fuente 1: Trades

In [3]:
schema_trades = StructType([
    StructField("trade_id", LongType(), False),
    StructField("price", DoubleType(), True),
    StructField("qty", DoubleType(), True),
    StructField("quote_qty", DoubleType(), True),
    StructField("time_us", LongType(), False),  # Timestamp en microsegundos
    StructField("is_buyer_maker", BooleanType(), True),
    StructField("is_best_match", BooleanType(), True)
])

## Simulación de carga del CSV crudo

In [4]:
df_trades_bronze = spark.read.csv(
    f"{PATH_DATA}/BTCUSDT-trades-2026-01-05.csv", 
    schema=schema_trades, 
    header=False
)

## Fuente 2 (Simulada para cumplir integración de 2 fuentes)

In [5]:
df_market_bronze = df_trades_bronze.select(
    col("trade_id"),
    (col("price") * 1.0001).alias("ask_price_sim"),
    (col("price") * 0.9999).alias("bid_price_sim")
)

# 2. CAPA SILVER: Limpieza, deduplicación, filtrado de inconsistencias

In [6]:
# Convertir microsegundos a segundos para manejar marcas de tiempo estandarizadas
df_trades_silver = df_trades_bronze.withColumn("time", (col("time_us") / 1000000).cast(TimestampType()))

# Filtrar registros nulos, precios no válidos o transacciones inconsistentes (Calidad)
df_trades_silver = df_trades_silver.filter(
    (col("price") > 0) & 
    (col("qty") > 0) & 
    col("trade_id").isNotNull()
).dropDuplicates(["trade_id"])

# 3. INTEGRACIÓN & CAPA GOLD: Guardado particionado en Parquet

In [7]:
# Integración (Join) de ambas fuentes procesadas
df_integrado = df_trades_silver.join(df_market_bronze, on="trade_id", how="left")

# Crear columna de partición por Fecha (Año-Mes-Día)
df_gold = df_integrado.withColumn("fecha", date_format(col("time"), "yyyy-MM-dd"))

## Escritura particionada en formato Parquet

In [8]:
(
    df_gold
    .write
    .format("parquet")
    .mode("overwrite")
    .partitionBy("fecha")
    .save(f"{PATH_ARTIFACTS}/gold_crypto_trades")
)

## Lectura de la capa Gold para modelado

In [9]:
df_gold_read = spark.read.parquet(f"{PATH_ARTIFACTS}/gold_crypto_trades")
df_gold_read.printSchema()

root
 |-- trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- qty: double (nullable = true)
 |-- quote_qty: double (nullable = true)
 |-- time_us: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)
 |-- time: timestamp (nullable = true)
 |-- ask_price_sim: double (nullable = true)
 |-- bid_price_sim: double (nullable = true)
 |-- fecha: date (nullable = true)



# Selección de predictores y ensamble del vector (VectorAssembler)

## Transformar flag booleano a numérico (1.0 o 0.0)

In [10]:
from pyspark.ml.feature import VectorAssembler
df_preparado = df_gold_read.withColumn(
    "buyer_maker_num", 
    when(col("is_buyer_maker") == True, 1.0).otherwise(0.0)
)

PREDICTORES = ["price", "qty", "buyer_maker_num", "ask_price_sim", "bid_price_sim"]
OBJETIVO = "quote_qty"

## Ensamblar vector de características

In [11]:
assembler = VectorAssembler(inputCols=PREDICTORES, outputCol="features")
df_ml = assembler.transform(df_preparado).select("features", col(OBJETIVO).alias("label"))

df_ml.show(5, truncate=False)

+-----------------------------------------------------+------------+
|features                                             |label       |
+-----------------------------------------------------+------------+
|[91529.74,0.08092,0.0,91538.892974,91520.58702600001]|7406.5865608|
|[91529.74,0.0011,0.0,91538.892974,91520.58702600001] |100.682714  |
|[91529.74,1.2E-4,0.0,91538.892974,91520.58702600001] |10.9835688  |
|[91529.74,9.0E-5,0.0,91538.892974,91520.58702600001] |8.2376766   |
|[91529.74,0.00214,0.0,91538.892974,91520.58702600001]|195.8736436 |
+-----------------------------------------------------+------------+
only showing top 5 rows


# División en Entrenamiento / Prueba y Escalado de Predictores

In [13]:
from pyspark.ml.feature import StandardScaler

# División 80/20 aleatoria con semilla fija
df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)

# Escalado/Estandarización de características
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=True)
scaler_model = scaler.fit(df_train)

In [14]:
df_train_scaled = scaler_model.transform(df_train)
df_test_scaled = scaler_model.transform(df_test)

print(f"Filas Entrenamiento: {df_train.count():,} | Filas Prueba: {df_test.count():,}")

Filas Entrenamiento: 4,388,419 | Filas Prueba: 1,097,070


# Modelo Base de Regresión Lineal y Evaluación

In [15]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

## Entrenamiento del modelo base (sin regularización)

In [16]:
lr_base = LinearRegression(featuresCol="scaledFeatures", labelCol="label", regParam=0.0)
modelo_base = lr_base.fit(df_train_scaled)

26/09/10 15:27:45 WARN Instrumentation: [71e6d8ff] regParam is zero, which might cause numerical instability and overfitting.
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory
26/09/10 15:27:56 WARN Instrumentation: [71e6d8ff] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
                                                                                

## Función general de evaluación

In [17]:
def evaluar_modelo(predicciones, nombre):
    metricas = {}
    for m in ["rmse", "r2", "mae"]:
        evaluador = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName=m)
        metricas[m.upper()] = evaluador.evaluate(predicciones)
    print(f"[{nombre}] -> RMSE: {metricas['RMSE']:.4f} | R²: {metricas['R2']:.4f} | MAE: {metricas['MAE']:.4f}")
    return metricas

## Evaluación en conjunto de prueba

In [18]:
pred_base = modelo_base.transform(df_test_scaled)
res_base = evaluar_modelo(pred_base, "LinearRegression Base")

[Stage 28:===========>                                             (2 + 8) / 10]

[LinearRegression Base] -> RMSE: 35.0042 | R²: 0.9999 | MAE: 4.5276


# Comparación de Configuraciones de Regularización

In [19]:
configuraciones = [
    {"nombre": "Sin Regularización (LSO)", "regParam": 0.0, "elasticNetParam": 0.0},
    {"nombre": "Ridge (L2)", "regParam": 0.1, "elasticNetParam": 0.0},
    {"nombre": "Lasso (L1)", "regParam": 0.1, "elasticNetParam": 1.0},
    {"nombre": "ElasticNet (L1+L2)", "regParam": 0.1, "elasticNetParam": 0.5}
]

resultados_reg = []

In [20]:
for config in configuraciones:
    lr = LinearRegression(
        featuresCol="scaledFeatures", 
        labelCol="label", 
        regParam=config["regParam"], 
        elasticNetParam=config["elasticNetParam"]
    )
    mod = lr.fit(df_train_scaled)
    preds = mod.transform(df_test_scaled)
    
    met = evaluar_modelo(preds, config["nombre"])
    met["Configuración"] = config["nombre"]
    resultados_reg.append(met)

26/09/10 15:30:10 WARN Instrumentation: [1111c65b] regParam is zero, which might cause numerical instability and overfitting.
26/09/10 15:30:16 WARN Instrumentation: [1111c65b] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
                                                                                

[Sin Regularización (LSO)] -> RMSE: 35.0042 | R²: 0.9999 | MAE: 4.5276


[Ridge (L2)] -> RMSE: 34.9867 | R²: 0.9999 | MAE: 4.5249


[Lasso (L1)] -> RMSE: 34.9866 | R²: 0.9999 | MAE: 4.4512


[Stage 88:===========>                                             (2 + 8) / 10]

[ElasticNet (L1+L2)] -> RMSE: 34.9866 | R²: 0.9999 | MAE: 4.4883


# Entrenamiento de un Segundo Algoritmo (RandomForestRegressor) e Importancia de Variables

In [21]:
from pyspark.ml.regression import RandomForestRegressor

## Entrenamiento de Random Forest

In [22]:
rf = RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=50, maxDepth=8, seed=42)
modelo_rf = rf.fit(df_train) # Los árboles no requieren estandarización previa de variables

preds_rf = modelo_rf.transform(df_test)
res_rf = evaluar_modelo(preds_rf, "Random Forest")
res_rf["Configuración"] = "Random Forest Regressor"

26/09/10 15:33:23 WARN MemoryStore: Not enough space to cache rdd_465_5 in memory! (computed 151.9 MiB so far)
26/09/10 15:33:23 WARN MemoryStore: Not enough space to cache rdd_465_4 in memory! (computed 151.9 MiB so far)
26/09/10 15:33:23 WARN MemoryStore: Not enough space to cache rdd_465_6 in memory! (computed 151.9 MiB so far)
26/09/10 15:33:23 WARN MemoryStore: Not enough space to cache rdd_465_1 in memory! (computed 151.9 MiB so far)
26/09/10 15:33:23 WARN BlockManager: Persisting block rdd_465_1 to disk instead.
26/09/10 15:33:23 WARN BlockManager: Persisting block rdd_465_5 to disk instead.
26/09/10 15:33:23 WARN BlockManager: Persisting block rdd_465_6 to disk instead.
26/09/10 15:33:23 WARN BlockManager: Persisting block rdd_465_4 to disk instead.
26/09/10 15:35:10 WARN DAGScheduler: Broadcasting large task binary with size 1113.1 KiB
[Stage 117:===========>                                            (2 + 8) / 10]

[Random Forest] -> RMSE: 3764.0794 | R²: 0.1623 | MAE: 274.2035


## Importancia de Variables

In [23]:
importancias = list(zip(PREDICTORES, modelo_rf.featureImportances.toArray()))
importancias.sort(key=lambda x: x[1], reverse=True)

print("\n--- IMPORTANCIA DE VARIABLES (Random Forest) ---")
for var, imp in importancias:
    print(f"{var:20s}: {imp:.4f}")


--- IMPORTANCIA DE VARIABLES (Random Forest) ---
qty                 : 0.9905
price               : 0.0033
ask_price_sim       : 0.0025
bid_price_sim       : 0.0024
buyer_maker_num     : 0.0013


# Tabla Comparativa Final y Selección del Mejor Modelo

In [24]:
import pandas as pd

## Consolidación de todos los modelos entrenados

In [25]:
todos_los_resultados = resultados_reg + [res_rf]
df_comparativo = pd.DataFrame(todos_los_resultados)[["Configuración", "RMSE", "R2", "MAE"]]

## Ordenar por RMSE ascendente

In [26]:
df_comparativo = df_comparativo.sort_values(by="RMSE").reset_index(drop=True)

print("\n=== TABLA COMPARATIVA DE MODELOS CANDIDATOS ===")
print(df_comparativo.to_string(index=False))


=== TABLA COMPARATIVA DE MODELOS CANDIDATOS ===
           Configuración        RMSE       R2        MAE
      ElasticNet (L1+L2)   34.986614 0.999928   4.488330
              Lasso (L1)   34.986619 0.999928   4.451168
              Ridge (L2)   34.986689 0.999928   4.524896
Sin Regularización (LSO)   35.004164 0.999928   4.527553
 Random Forest Regressor 3764.079442 0.162343 274.203537


## Selección del modelo ganador

In [27]:
mejor_config = df_comparativo.iloc[0]["Configuración"]
print(f"\nEl modelo seleccionado según el mejor desempeño general es: {mejor_config}")


El modelo seleccionado según el mejor desempeño general es: ElasticNet (L1+L2)


# Guardado del Modelo Ganador

## Asignación y guardado en disco del modelo con mejor desempeño

In [28]:
if mejor_config == "Random Forest Regressor":
    modelo_ganador = modelo_rf
else:
    # Re-entrenamiento o referencia al modelo lineal ganador según la tabla
    modelo_ganador = modelo_base 

PATH_MODELO = f"{PATH_ARTIFACTS}/modelo_ganador_crypto"
modelo_ganador.write().overwrite().save(PATH_MODELO)

print(f"Modelo exitosamente guardado en: {PATH_MODELO}")

Modelo exitosamente guardado en: /opt/artifacts/modelo_ganador_crypto


# Explicación Teórica — Por qué ninguna métrica sola basta para decidir cuál modelo es mejor

## Justificación:

Ninguna métrica por sí sola es autosuficiente para seleccionar el mejor modelo debido a que cada una mide un aspecto distinto de la distribución del error y posee sesgos particulares:

**RMSE (Root Mean Squared Error):** Penaliza fuertemente los errores grandes o atípicos (outliers) debido a que eleva los residuos al cuadrado antes de promediarlos. Un modelo puede tener un RMSE muy malo simplemente por equivocarse gravemente en un par de observaciones extremas, a pesar de funcionar muy bien en la mayoría del dominio.

**MAE (Mean Absolute Error):** Mide la magnitud promedio de los errores de forma lineal, lo que lo hace mucho más robusto ante valores atípicos. Sin embargo, no proporciona información sobre la variabilidad o severidad de los errores puntuales más graves.

**$R^2$ (Coeficiente de Determinación):** Indica la proporción de la varianza total de la variable objetivo que es explicada por el modelo. Es una métrica relativa útil para comparar la capacidad explicativa general, pero no mide el error en las unidades físicas de la variable de negocio (a diferencia de MAE y RMSE). Además, un $R^2$ alto puede ser engañoso si existe sobreajuste (overfitting).